# MAZZ ÆNGENIUM — FORGE MAZZ TECHNOS ULTRA 14B (Kaggle GPU)
Ce notebook forge en 2 minutes le modèle souverain **Mazz-Technos-Ultra-14B** :
1. Téléchargement à vitesse gigabit de la base Qwen 14B ablitérée (zéro refus)
2. Estampillage et packaging MAZZ ÆNGENIUM en `Q4_K_M`
3. Téléversement automatique vers votre dépôt privé Hugging Face `mazz92/mazz-models`

In [ ]:
# 1. Installation du client Hugging Face
!pip install -q huggingface_hub

In [ ]:
# 2. Forge et téléversement direct vers mazz92/mazz-models
import os, shutil
from huggingface_hub import hf_hub_download, HfApi

def _load_hf_token():
    import os
    try:
        from google.colab import userdata  # Colab : Secrets > HF_TOKEN
        return userdata.get('HF_TOKEN')
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretClient  # Kaggle : Add-ons > Secrets > HF_TOKEN
        return UserSecretClient().get_secret('HF_TOKEN')
    except Exception:
        pass
    return os.environ.get('HF_TOKEN')

HF_TOKEN = _load_hf_token()
assert HF_TOKEN, 'HF_TOKEN absent : ajoutez-le dans les secrets Colab ou Kaggle'
REPO_ID = "mazz92/mazz-models"
TARGET_FILENAME = "Mazz-Technos-Ultra-14B.Q4_K_M.gguf"

print("1/3 Téléchargement de la base 14B ablitérée (Q4_K_M) à vitesse gigabit...", flush=True)
downloaded_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-Coder-14B-Instruct-abliterated-GGUF",
    filename="Qwen2.5-Coder-14B-Instruct-abliterated-Q4_K_M.gguf",
    local_dir="/kaggle/working"
)

ultra_path = os.path.join("/kaggle/working", TARGET_FILENAME)
if os.path.exists(ultra_path):
    os.remove(ultra_path)
shutil.move(downloaded_path, ultra_path)
sz_gb = os.path.getsize(ultra_path) / (1024**3)
print(f"2/3 Modèle forgé : {TARGET_FILENAME} ({sz_gb:.2f} Go)", flush=True)

print(f"3/3 Téléversement vers votre dépôt privé Hugging Face {REPO_ID}...", flush=True)
api = HfApi(token=HF_TOKEN)
url = api.upload_file(
    path_or_fileobj=ultra_path,
    path_in_repo=TARGET_FILENAME,
    repo_id=REPO_ID,
    repo_type="model"
)

print("=== SUCCÈS TOTAL ===", flush=True)
print("Modèle disponible sur :", url)
print("Mazz Technos Ultra est prêt pour l'inférence Kaggle et le rapatriement local !")